# DataFrames: load and inspect ShopKart

**Video 6.1** | ShopKart project

**Goal:** Load the three ShopKart files into pandas and use head, shape, info, describe and isnull to find every problem before we touch anything.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_raw.csv`, `customers.csv`, `products.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

## 1. Import pandas and load the files
A **DataFrame** is a table: rows, columns, and an index (row labels on the left). A single column is a **Series**.

In [1]:
import pandas as pd

orders = pd.read_csv("orders_raw.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

## 2. First look: head() and tail()

In [2]:
orders.head()

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,city,payment_mode,delivery_days,rating,returned
0,O10001,2025-01-01,C0593,P048,6,362,0.20,Delhi,COD,4,2.0,N
1,O10002,2025-01-01,C0367,P049,2,300,0.05,Chennai,Debit Card,4,5.0,N
2,O10003,2025-01-01,C0707,P020,1,2258,0.05,DELHI,Debit Card,2,3.0,N
3,O10004,2025-01-01,C0118,P026,2,2020,0.15,Lucknow,COD,4,4.0,Y
4,O10005,2025-01-01,C0238,P026,2,1937,0.10,Chennai,UPI,2,4.0,N


In [3]:
orders.tail(3)

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,city,payment_mode,delivery_days,rating,returned
5052,O14998,2025-12-28,C0753,P042,2,980,0.05,Hyderabad,Net Banking,2,5.0,N
5053,O14999,2025-12-28,C0579,P043,1,596,0.15,Mumbai,UPI,2,4.0,N
5054,O15000,2025-12-28,C0326,P002,2,2499,0.10,Lucknow,Debit Card,7,2.0,Y


## 3. How big? shape and columns

In [4]:
print("orders:", orders.shape)
print("customers:", customers.shape)
print("products:", products.shape)
print(list(orders.columns))

orders: (5055, 12)
customers: (800, 7)
products: (60, 5)
['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount', 'city', 'payment_mode', 'delivery_days', 'rating', 'returned']


5,055 order rows - but ShopKart had 5,000 orders in 2025. Remember that.

## 4. Column types: info()
`object` / `str` = text, `int64` = whole numbers, `float64` = decimals. Look at **unit_price** and **order_date**.

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 5055 entries, 0 to 5054
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5055 non-null   str    
 1   order_date     5055 non-null   str    
 2   customer_id    5055 non-null   str    
 3   product_id     5055 non-null   str    
 4   quantity       5055 non-null   int64  
 5   unit_price     5055 non-null   str    
 6   discount       4985 non-null   float64
 7   city           5055 non-null   str    
 8   payment_mode   4965 non-null   str    
 9   delivery_days  5055 non-null   int64  
 10  rating         4792 non-null   float64
 11  returned       5055 non-null   str    
dtypes: float64(2), int64(2), str(8)
memory usage: 474.0 KB


**unit_price is text**, not a number - because some rows hold values like `"₹1,299"`. **order_date is text** too.
The *Non-Null Count* column already tells us discount, payment_mode and rating have gaps.

## 5. Number summary: describe()

In [6]:
orders.describe()

,quantity,discount,delivery_days,rating
count,5055.000000,4985.000000,5055.000000,4792.000000
mean,1.868249,0.117553,4.158655,3.957846
std,2.889888,0.095640,1.533994,0.824571
min,1.000000,0.000000,2.000000,1.000000
25%,1.000000,0.050000,3.000000,4.000000
50%,2.000000,0.100000,4.000000,4.000000
75%,2.000000,0.200000,5.000000,5.000000
max,99.000000,0.400000,12.000000,5.000000


Look at **quantity**: the 75% value is 2 but the max is **99**. Nobody orders 99 face washes. Suspicious.

In [7]:
# Which rows have the huge quantities?
orders["quantity"].value_counts().sort_index()

quantity
1     2334
2     1797
3      663
4      184
5       55
6       15
10       3
99       4
Name: count, dtype: int64

## 6. Missing values: isnull().sum()

In [8]:
orders.isnull().sum()

order_id           0
order_date         0
customer_id        0
product_id         0
quantity           0
unit_price         0
discount          70
city               0
payment_mode      90
delivery_days      0
rating           263
returned           0
dtype: int64

## 7. Text columns: how many spellings?

In [9]:
print("Distinct city spellings:", orders["city"].nunique())
orders["city"].value_counts()

Distinct city spellings: 27


city
Delhi         715
Mumbai        679
Bengaluru     651
Hyderabad     441
Pune          403
Chennai       392
Kolkata       369
Lucknow       331
Ahmedabad     304
Jaipur        276
hyderabad      59
Bangalore      40
 Mumbai        35
mumbai         34
New Delhi      32
Pune           31
chennai        30
Calcutta       27
Delhi          27
bengaluru      26
DELHI          24
kolkata        23
pune           23
Madras         23
Bengaluru      23
Bombay         22
delhi          15
Name: count, dtype: int64

In [10]:
print(orders["payment_mode"].value_counts(dropna=False))
print(orders["returned"].value_counts())

payment_mode
UPI            2014
COD             998
Credit Card     896
Debit Card      693
Net Banking     304
NaN              90
upi              60
Name: count, dtype: int64
returned
N      4292
Y       683
No       68
Yes      12
Name: count, dtype: int64


In [11]:
# Exact duplicate rows (every column identical)
orders.duplicated().sum()

np.int64(55)

In [12]:
# Prices stored as text with the rupee symbol, and dates in DD-MM-YYYY
print(orders.loc[orders["unit_price"].str.contains("₹"), "unit_price"].head())
print(orders.loc[orders["order_date"].str.contains(r"^\d{2}-"), "order_date"].head())

28       ₹626
40     ₹2,551
67       ₹981
109    ₹2,563
111    ₹1,540
Name: unit_price, dtype: str
15    02-01-2025
19    02-01-2025
33    03-01-2025
67    06-01-2025
98    09-01-2025
Name: order_date, dtype: str


## 8. The other two tables look clean

In [13]:
print(customers.isnull().sum().sum(), "missing values in customers")
print(products.isnull().sum().sum(), "missing values in products")
customers.head()

0 missing values in customers
0 missing values in products


,customer_id,name,gender,age,city,signup_date,membership
0,C0001,Neha Kapoor,Female,18,Kolkata,2023-05-31,Free
1,C0002,Sneha Das,Female,36,Delhi,2023-01-16,Prime
2,C0003,Rohit Kulkarni,Male,20,Delhi,2024-07-23,Prime
3,C0004,Nisha Khan,Female,26,Kolkata,2024-12-26,Free
4,C0005,Priya Khan,Female,27,Delhi,2022-07-02,Free


In [14]:
products["category"].value_counts()

category
Electronics    12
Fashion        12
Home           12
Books          12
Beauty         12
Name: count, dtype: int64

## Problem list (our to-do for video 6.3)
| # | Problem | How we found it |
|---|---|---|
| 1 | 55 exact duplicate rows | `duplicated().sum()` |
| 2 | 27 spellings of 10 cities | `value_counts()` |
| 3 | unit_price is text (₹ and commas) | `info()` |
| 4 | order_date in two formats | `info()` + looking |
| 5 | blank discount, blank payment_mode, "upi" lowercase | `isnull()`, `value_counts()` |
| 6 | returned has Yes/No mixed with Y/N | `value_counts()` |
| 7 | quantity 99 | `describe()` |
| 8 | blank rating | `isnull()` - but this one is real (customer did not rate) |

## Practice
How many customers are Prime and how many Free? What is the cheapest and most expensive product by list_price?

In [15]:
print(customers["membership"].value_counts())
print(products.sort_values("list_price").iloc[[0, -1]][["product_name", "list_price"]])

membership
Free     511
Prime    289
Name: count, dtype: int64
   product_name  list_price
57        Kajal         199
9        Tablet       18999
